# Rejection-Sampling Dataset Generation (Qwen2.5-Coder-7B self-distillation)

**Mục đich:** tạo bộ data fine-tune nhỏ, có mục tiêu, để sửa đúng loại lỗi (logic/edge-case) mà `Qwen2.5-Coder-7B` base đang mắc trên HumanEval/HumanEval+ (xem phân tích fail case trước đó).

**Nguồn problem pool: MBPP `train` + `validation` split only (464 problems).** KHÔNG dùng MBPP `test` split, và tuyệt đối không dùng HumanEval/HumanEval+ — để tránh contamination với benchmark đang dùng để đánh giá.

**Cơ chế (rejection sampling / RFT):**
1. Cho chính model hiện tại tự sinh `N_SAMPLES` lời giải/problem (temperature cao để đa dạng).
2. Chạy unit test thật (test_list của MBPP) để lọc — chỉ giữ sample PASS.
3. Nếu không sample nào pass (model luôn sai problem đó), fallback dùng canonical solution của MBPP (đánh dấu `canonical_fallback=true`) để vẫn có example cho problem đó, tránh bỏ sót toàn bộ nhóm khó.
4. Format lại thành (instruction, output) giống schema dataset magicoder cũ, để dùng được ngay cho `fine-tune-qwen-unsloth.ipynb`.

**Lưu ý:** notebook này viết để chạy trên Kaggle, nhưng theo yêu cầu KHÔNG chạy thật ở đây — dự đoán kết quả (pass rate, kích thước dataset cuối, thời gian) được trình bày riêng trong phần trả lời, không nằm trong notebook.

In [ ]:
!pip install -q datasets transformers bitsandbytes accelerate sentencepiece huggingface_hub tqdm

## 1. Config

In [ ]:
import gc
import json
import re
import multiprocessing
import time
from tqdm import tqdm
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from datasets import load_dataset, concatenate_datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B"
N_SAMPLES = 8          # samples per problem (rejection sampling budget)
TEMPERATURE = 0.7       # higher temp than eval -> more diversity to sample from
TOP_P = 0.95
MAX_NEW_TOKENS = 400
MAX_KEEP_PER_PROBLEM = 2   # cap passing samples kept per problem (avoid over-weighting easy ones)
EXEC_TIMEOUT = 5
OUTPUT_FILE = "rft_dataset.jsonl"
HUB_DATASET_ID = "hoangnh39/qwen25coder-mbpp-rft"  # optional push target

## 2. Problem Pool (MBPP train + validation, test split excluded)

In [ ]:
mbpp = load_dataset("google-research-datasets/mbpp", "full")
problem_pool = concatenate_datasets([mbpp["train"], mbpp["validation"]])
print(f"Problem pool size: {len(problem_pool)} (train={len(mbpp['train'])}, validation={len(mbpp['validation'])})")
print("NOTE: mbpp['test'] (500 problems) is intentionally excluded from this pool.")

## 3. Prompt Template, Code Extraction, Execution Harness

In [ ]:
ENTRY_POINT_RE = re.compile(r"assert\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(")

def extract_entry_point(test_list):
    match = ENTRY_POINT_RE.search(test_list[0])
    return match.group(1) if match else None

GEN_PROMPT = """You are an expert Python programmer. Write the implementation for the following problem.
Do not output anything else but the code block.

Requirements:
- Output only a single Python code block.
- Include all necessary imports explicitly.
- The function must be named exactly `{entry_point}`.
- The solution must be self-contained and executable.

Problem:
{text}

Code:
"""

def build_prompt(row, entry_point):
    return GEN_PROMPT.format(text=row["text"], entry_point=entry_point)

def clean_code(generated_text):
    match = re.search(r"```python\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match:
        return match.group(1)
    match = re.search(r"```\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match:
        return match.group(1)
    return generated_text.strip()

def run_eval(code, setup_code, test_list, queue):
    local_scope = {}
    try:
        if setup_code:
            exec(setup_code, local_scope)
        exec(code, local_scope)
        for assertion in test_list:
            exec(assertion, local_scope)
        queue.put(True)
    except Exception:
        queue.put(False)

def exec_code(code, setup_code, test_list, timeout=EXEC_TIMEOUT):
    queue = multiprocessing.Queue()
    p = multiprocessing.Process(target=run_eval, args=(code, setup_code, test_list, queue))
    p.start()
    p.join(timeout)
    if p.is_alive():
        p.terminate()
        p.join()
        return False
    return queue.get() if not queue.empty() else False

## 4. Load Model (4-bit, same config as eval pipeline)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:,.1f} GB")

## 5. Generation + Rejection Filtering Loop

In [ ]:
def generate_samples(prompt, n_samples=N_SAMPLES):
    """Batched generation: num_return_sequences shares the prompt prefill across all n samples
    (much faster than looping generate() n_samples times)."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    try:
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            num_return_sequences=n_samples,
            pad_token_id=tokenizer.eos_token_id
        )
    except Exception as e:
        print(f"Generation error: {e}")
        return []

    prompt_length = inputs["input_ids"].shape[1]
    return [
        tokenizer.decode(out[prompt_length:], skip_special_tokens=True)
        for out in outputs
    ]

records = []
stats = {"total": 0, "solved_by_sampling": 0, "canonical_fallback": 0, "unparseable_entry_point": 0}

for row in tqdm(problem_pool):
    entry_point = extract_entry_point(row["test_list"])
    if entry_point is None:
        stats["unparseable_entry_point"] += 1
        continue

    stats["total"] += 1
    prompt = build_prompt(row, entry_point)
    raw_samples = generate_samples(prompt)

    passing_codes = []
    seen = set()
    for raw in raw_samples:
        code = clean_code(raw)
        if code in seen:
            continue
        seen.add(code)
        if exec_code(code, row["test_setup_code"], row["test_list"]):
            passing_codes.append(code)
        if len(passing_codes) >= MAX_KEEP_PER_PROBLEM:
            break

    if passing_codes:
        stats["solved_by_sampling"] += 1
        for code in passing_codes:
            records.append({
                "task_id": row["task_id"],
                "instruction": prompt,
                "output": f"```python\n{code}\n```",
                "canonical_fallback": False
            })
    else:
        stats["canonical_fallback"] += 1
        records.append({
            "task_id": row["task_id"],
            "instruction": prompt,
            "output": f"```python\n{row['code']}\n```",
            "canonical_fallback": True
        })

print(stats)
print(f"Total training examples: {len(records)}")

## 6. Save Dataset

In [ ]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved {len(records)} examples to {OUTPUT_FILE}")

# Optional: push to Hugging Face Hub for use in fine-tune-qwen-unsloth.ipynb
# from datasets import Dataset
# Dataset.from_list(records).push_to_hub(HUB_DATASET_ID, private=True)